# 第 1 课：高级 RAG 流水线

In [2]:
import llama_index.core # 关键改变：引入 core
from trulens.apps.llamaindex import TruLlama
import utils
import importlib
importlib.reload(utils)

# 从 core 中读取版本号
print(f"📦 当前 LlamaIndex 版本: {llama_index.core.__version__}")
print("🎯 正在测试 TruLlama 初始化...")

try:
    from utils import liteLLM_provider
    print("✅ 恭喜！环境彻底通关，LiteLLM 已加载。")
except Exception as e:
    print(f"⚠️ 仍然存在小问题: {e}")

📦 当前 LlamaIndex 版本: 0.14.21
🎯 正在测试 TruLlama 初始化...
✅ 恭喜！环境彻底通关，LiteLLM 已加载。


In [1]:
import importlib
import utils
importlib.reload(utils)

import os
import openai

In [2]:
from llama_index.core import SimpleDirectoryReader  # 导入 LlamaIndex 核心库中的简单目录读取器

# 使用 SimpleDirectoryReader 实例化一个加载器，并明确指定要加载的 PDF 文件路径
documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI-中文.pdf"]
).load_data()  # 执行加载操作，返回一个 Document 对象列表

In [3]:
print(type(documents), "\n")
print(len(documents), "\n")
print(type(documents[0]))
print(documents[0])

<class 'list'> 

41 

<class 'llama_index.core.schema.Document'>
Doc ID: 53ca321a-e94f-40b9-b928-6b5c99fc1dec
Text: P AGE 1 Founder , DeepLearning.AI C o l l e ct e d  I n s i g h
t s f r o m  A n d r e w  N g How to Build Y our Care e r i n AI A
Simple Guide 第 1  ⻚ DeepL earning.AI 创始⼈ 收集到的⻅解 来⾃ Andrew Ng 如何 建造 你的
职业 ⼈⼯智能 简易指南


## 基础 RAG 流水线

In [4]:
from llama_index.core import Document

# 文档合并: PDF 文件可能会被分割成多个 Document 对象（例如按页）。
# 为了简化处理，代码将所有这些 Document 的文本内容通过换行符 \n\n 连接起来，
# 创建了一个单独的、包含全部文本的 Document 对象
document = Document(text="\n\n".join([doc.text for doc in documents]))

In [5]:
import os
# 设置环境变量，将 HuggingFace 的默认地址指向镜像站（hf-mirror.com）
# 这通常是为了解决某些地区访问 HuggingFace 官方服务器缓慢或无法连接的问题
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from llama_index.core import VectorStoreIndex  # 用于构建和查询向量索引的核心类
from llama_index.core import Settings  # LlamaIndex 的全局配置对象，用于设置 LLM、Embedding 等
from llama_index.llms.openai_like import OpenAILike  # 用于连接兼容 OpenAI 接口标准的自定义 LLM 服务
from llama_index.embeddings.huggingface import HuggingFaceEmbedding  # 用于加载本地或远程的 HuggingFace 嵌入模型
from llama_index.core.utils import get_cache_dir  # 获取 LlamaIndex 默认缓存目录的工具函数

# 查看并自定义本机缓存目录
# 模型（尤其是 Embedding 模型）文件很大，我们需要明确它们下载到了哪里
cache_folder = os.path.join(get_cache_dir(), "models")
os.makedirs(cache_folder, exist_ok=True)  # 如果该文件夹不存在，则自动创建它

print(f"LlamaIndex embed_model缓存路径: {cache_folder}")


# OpenAILike: LlamaIndex 中用于连接与 OpenAI API 兼容的服务的类
# 这里用于连接阿里云的通义千问 (DashScope) 服务
#    配置了 API Key、基础 URL (api_base) 和模型 (qwen-max)。
#    设置了较低的 temperature=0.1，以获得更确定性的回答。
#    设置了较大的 context_window=128000。
llm = OpenAILike(
    api_key=utils.get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=False,
)

# Settings: LlamaIndex 的全局配置对象
# 将全局的语言模型设置为配置好的 OpenAILike 实例
Settings.llm = llm
# 将全局的嵌入模型设置为本地加载的 BGE-Small 模型，用于将文本内容转化为向量
# 本地且无法连接外网的时候，可以指定目录
model_real_path = os.path.expanduser(
    "~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
Settings.embed_model = HuggingFaceEmbedding(
    model_name=model_real_path,
    # model_name="BAAI/bge-small-en-v1.5",
    # 如果您的机器有 GPU，建议设置 device="cuda"
    device="cpu", 
    # 连不了外网记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
    local_files_only=True,
)

# VectorStoreIndex.from_documents: 这是 LlamaIndex 中创建向量索引的核心步骤。
# 它使用全局配置的 embed_model，将前面合并的 document 切分（默认按块）并嵌入成向量，存储在一个向量存储中。
index = VectorStoreIndex.from_documents([document])

2026-04-23 22:48:24,386 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a


LlamaIndex embed_model缓存路径: /Users/a1-6/Library/Caches/llama_index/models


In [6]:
# 从构建好的索引创建一个查询引擎 (query_engine)。
# 这个引擎负责接收用户的查询，进行检索（找到最相关的文本块），
# 并将检索到的上下文和查询一起发送给全局配置的 llm 以生成答案
query_engine = index.as_query_engine()

In [9]:
# 对查询引擎进行一次直接查询测试，打印出模型针对该问题的回答
response = query_engine.query(
    # "What are steps to take when finding projects to build your experience?"
    "在寻找项目来积累你的经验时，应该采取哪些步骤?"
)
print(str(response))

KeyboardInterrupt: 

## Evaluation setup using TruLens
## 使用 TruLens 进行评估设置

In [15]:
# 从名为 eval_questions.txt 的文件中读取一系列用于评估 RAG 系统性能的问答（QA）问题，并将它们存储在列表 eval_questions 中
eval_questions = []  # 初始化一个空列表，用来存放从文件里读取出来的所有问题
with open('eval_questions.txt', 'r') as file:  # 使用 with 语句（上下文管理器）打开文件。'r' 表示只读模式。
                                               # 使用 with 的好处是：即使读取出错，文件也会在结束时被自动关闭。
    for line in file:  # 遍历文件中的每一行
        # Remove newline character and convert to integer
        item = line.strip()  # 使用 .strip() 去掉行首和行尾的空白字符，最重要的是去掉换行符（\n）
        
        print(item)  # 将处理后的每一行（即每一个问题）打印到控制台，方便你实时查看进度
        eval_questions.append(item)  # 将清洗后的字符串添加到 eval_questions 列表中

在人工智能领域建立职业生涯的关键是什么？
团队合作如何有助于人工智能的成功？
网络在人工智能中的重要性是什么？
要想事业成功，有哪些好习惯？
利他主义如何有益于事业的发展？
什么是冒名顶替综合症？它与人工智能有什么关系？
有哪些成功人士经历过冒名顶替综合症？
精通人工智能的第一步是什么？
人工智能有哪些共同的挑战？
发现AI的某些部分具有挑战性是正常的吗？


In [16]:
# You can try your own question:
new_question = "什么是适合我的AI工作?"
eval_questions.append(new_question)

In [17]:
print(eval_questions)

['在人工智能领域建立职业生涯的关键是什么？', '团队合作如何有助于人工智能的成功？', '网络在人工智能中的重要性是什么？', '要想事业成功，有哪些好习惯？', '利他主义如何有益于事业的发展？', '什么是冒名顶替综合症？它与人工智能有什么关系？', '有哪些成功人士经历过冒名顶替综合症？', '精通人工智能的第一步是什么？', '人工智能有哪些共同的挑战？', '发现AI的某些部分具有挑战性是正常的吗？', '什么是适合我的AI工作?']


In [11]:
from trulens.core import Tru  # 从 trulens 核心库中导入 Tru 类，它是 TruLens 的管理中枢
# 创建 Tru 类的实例对象
# 这一步会初始化 TruLens 的后端数据库（默认是本地的 SQLite 文件），用于存储评估结果、反馈分数和调用记录
tru = Tru()

# 重置数据库，删除所有之前存储的评估数据
# 在实验初期非常有用，它可以帮你清空旧的测试记录，确保接下来的评估数据是干净的，不会与之前的实验混淆
tru.reset_database()

/var/folders/zt/nbrxt1lj4g7gf5l2hqmsq4m00000gp/T/ipykernel_19148/3758496272.py:4: DeprecationWarning: Tru is deprecated, use TruSession instead.
  tru = Tru()
2026-04-23 23:37:38,241 - INFO - Context impl SQLiteImpl.
2026-04-23 23:37:38,242 - INFO - Will assume non-transactional DDL.


2026-04-23 23:37:38,272 - INFO - ✅ OpenTelemetry exporter set: NoneType
2026-04-23 23:37:38,331 - INFO - ✅ Added new TrulensOtelSpanProcessor
2026-04-23 23:37:38,374 - INFO - Context impl SQLiteImpl.
2026-04-23 23:37:38,374 - INFO - Will assume non-transactional DDL.


🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.
✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


For the classroom, we've written some of the code in helper functions inside a utils.py file.  
- You can view the utils.py file in the file directory by clicking on the "Jupyter" logo at the top of the notebook.
- In later lessons, you'll get to work directly with the code that's currently wrapped inside these helper functions, to give you more options to customize your RAG pipeline.

对于课堂练习，我们已经将部分代码写在一个名为 utils.py 的辅助函数文件中。
- 你可以点击笔记本顶部的 “Jupyter” 标志，在文件目录中查看这个 utils.py 文件。
- 在后续的课程中，你将有机会直接操作目前封装在这些辅助函数中的代码，从而让你在自定义 RAG 流水线时拥有更多选择。

In [25]:
from utils import get_prebuilt_trulens_recorder

# 创建一个 TruLens 记录器
# query_engine 在执行查询时能自动记录输入（问题）、输出（答案）、检索到的上下文
# 以及各种内置的反馈指标（如相关性、一致性等）
tru_recorder = get_prebuilt_trulens_recorder(query_engine,
                                             app_id="Direct Query Engine")

instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.embeddings.multi_modal_base.MultiModalEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base

/Users/a1-6/miniconda3/envs/ailearn/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 5 fields but got 4: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='{\n  "cr...one, function_call=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ne, function_call=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/Users/a1-6/miniconda3/envs/ailearn/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 5 fields but got 4: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='{\n  "cr...one, function_call=None), input_type=Mess

In [26]:
# 在这个上下文管理器中执行的所有测试问题
with tru_recorder as recording:
    for question in eval_questions:
        response = query_engine.query(question)

In [27]:
# 从 TruLens 数据库中获取所有评估记录和计算的反馈得分
records, feedback = tru.get_records_and_feedback(app_ids=[])

In [28]:
# 打印出记录（一个 Pandas DataFrame）的前几行，展示了评估结果（如问题、答案、以及计算出的各种 TruLens 指标得分）
records.head()

,app_id,app_name,app_version,app_json,type,record_id,input_id,input,output,tags,...,cost_currency,num_events,Answer Relevance,Answer Relevance_calls,Answer Relevance feedback cost in USD,Answer Relevance direction,Context Relevance,Context Relevance_calls,Context Relevance feedback cost in USD,Context Relevance direction
0,app_hash_036435501dfb9ca99cbc0c2fdf8b03fa,LlamaIndex_App,base,"{'app_name': 'LlamaIndex_App', 'app_version': ...",SPAN,e87514a9-78d0-4ace-8e42-e6d26e920a28,,在人工智能领域建立职业生涯的关键是什么？,在人工智能领域建立职业生涯的关键包括逐步积累相关知识和技能，通过实际项目来应用这些技能，并且...,,...,USD,13,1.0,"[{'span_type': 'eval', 'args': {'prompt': '在人工...",0.0,True,0.5,"[{'span_type': 'eval', 'args': {'prompt': '在人工...",0.0,True
1,app_hash_036435501dfb9ca99cbc0c2fdf8b03fa,LlamaIndex_App,base,"{'app_name': 'LlamaIndex_App', 'app_version': ...",SPAN,2a64e5f4-8a11-42bf-9134-25550d174710,,团队合作如何有助于人工智能的成功？,团队合作在人工智能项目的成功中扮演着至关重要的角色。首先，构建有效的人工智能解决方案通常需要...,,...,USD,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,app_hash_036435501dfb9ca99cbc0c2fdf8b03fa,LlamaIndex_App,base,"{'app_name': 'LlamaIndex_App', 'app_version': ...",SPAN,29b8722b-1d07-4d7d-9f63-240fb0fc8baf,,网络在人工智能中的重要性是什么？,网络在人工智能中的重要性主要体现在它能够帮助收集和传输大量的数据，这些数据是训练人工智能模型...,,...,USD,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,app_hash_036435501dfb9ca99cbc0c2fdf8b03fa,LlamaIndex_App,base,"{'app_name': 'LlamaIndex_App', 'app_version': ...",SPAN,fd802fd6-9b1d-4ccc-b20b-1a3c9bf00e69,,要想事业成功，有哪些好习惯？,要想在事业上取得成功，可以培养以下几个好习惯：\n\n1. **持续学习**：技术领域尤其是...,,...,USD,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,app_hash_036435501dfb9ca99cbc0c2fdf8b03fa,LlamaIndex_App,base,"{'app_name': 'LlamaIndex_App', 'app_version': ...",SPAN,36694edd-a8fb-4b8d-9a43-16d8a62449ce,,利他主义如何有益于事业的发展？,利他主义可以通过多种方式有益于事业的发展。首先，通过帮助他人或公司解决他们的问题，即使这不是...,,...,USD,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# 启动一个本地的、交互式的 Web 用户界面（Dashboard），
# 用于可视化和探索您的 LLM 应用程序（RAG 引擎）的评估结果和性能数据
tru.run_dashboard()

Starting dashboard ...


/var/folders/zt/nbrxt1lj4g7gf5l2hqmsq4m00000gp/T/ipykernel_9286/3776832840.py:3: DeprecationWarning: Method `run_dashboard` has been renamed or moved to `trulens.dashboard.run.run_dashboard`.

  tru.run_dashboard()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:63680 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

In [30]:
# 停止当前运行的看板实例
tru.stop_dashboard()

/var/folders/zt/nbrxt1lj4g7gf5l2hqmsq4m00000gp/T/ipykernel_9286/244142309.py:2: DeprecationWarning: Method `stop_dashboard` has been renamed or moved to `trulens.dashboard.run.stop_dashboard`.

  tru.stop_dashboard()


## Advanced RAG pipeline （高级 RAG 流水线）

### 1. Sentence Window retrieval（句子窗口检索）

In [ ]:
# from llama_index.llms import OpenAI

# llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)

In [32]:
from utils import build_sentence_window_index

model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
# 使用自定义函数构建或加载 Sentence Window Index（句子窗口索引）。
# 这是 LlamaIndex 中一种优化检索的方法：检索小块（句子），但提供大块上下文（窗口）给 LLM。
sentence_index = build_sentence_window_index(
    document,
    llm,
    embed_model=model_real_path,
    save_dir="sentence_index"
)

In [33]:
from utils import get_sentence_window_query_engine

# 获取 Sentence Window 查询引擎。该引擎会自动执行 "检索句子+回填窗口上下文" 的逻辑。
sentence_window_engine = get_sentence_window_query_engine(sentence_index)

In [34]:
# 执行一次查询测试
window_response = sentence_window_engine.query(
    "我如何开始在人工智能的个人项目？"
)
print(str(window_response))

开始在人工智能领域的个人项目，你可以从一个小而简单的项目着手。例如，你可以尝试训练一个神经网络来模仿一个简单的数学函数，如sin(x)。这样的项目虽然可能没有实际的应用价值，但能为你提供宝贵的学习经验，并为将来更复杂的项目打下基础。

随着技能的提升，你可以逐渐增加项目的范围和复杂度。重要的是要能够清晰地沟通你的想法和成果，这样可以帮助你获得更多的资源和支持，从而开展更大规模的项目。此外，建立一个展示你技能逐步提升的项目组合，对于未来找工作也会非常有帮助。


In [ ]:
# 准备开始 Sentence Window 引擎的批量评估
# 先重置 TruLens 数据库，确保只记录当前策略的评估数据
tru.reset_database()

# 为 Sentence Window 查询引擎创建 TruLens 记录器
tru_recorder_sentence_window = get_prebuilt_trulens_recorder(
    sentence_window_engine,
    app_id = "Sentence Window Query Engine"
)

In [36]:
# 使用 TruLens 记录器对所有评估问题进行批量查询和评估
for question in eval_questions:
    with tru_recorder_sentence_window as recording:
        response = sentence_window_engine.query(question)
        print(question)
        print(str(response))

在人工智能领域建立职业生涯的关键是什么？
在人工智能领域建立职业生涯的关键包括几个方面：

1. **团队合作**：与他人协作、影响他人以及被他人影响的能力至关重要。因此，人际交往和沟通技巧非常重要。

2. **建立社区**：虽然有些人可能不喜欢传统的社交活动，但建立一个强大的专业网络可以在你需要帮助或建议时提供支持。可以考虑通过参与和建设你所在的社区来扩展人脉。

3. **求职**：找到工作只是职业生涯中的一个小步骤。重要的是要避免一些不好的求职建议，比如对潜在雇主采取对抗态度。相反，应该专注于长期的职业发展。

4. **个人自律**：成功的人往往会在饮食、锻炼、睡眠、人际关系、工作、学习和个人护理等方面养成良好的习惯。这些习惯有助于他们在保持健康的同时不断前进。

5. **利他主义**：在自己的职业旅程中帮助他人，往往会带来更好的结果。思考如何在追求自己职业目标的同时帮助他人。

这些关键点可以帮助你在人工智能领域建立一个成功且有意义的职业生涯。
团队合作如何有助于人工智能的成功？
在人工智能领域取得成功的过程中，团队合作起着至关重要的作用。当我们面对大型项目时，与他人协作比单打独斗更能取得好的成果。能够与他人合作、相互影响是非常关键的。因此，人际交往和沟通技巧非常重要。此外，在进行更大规模的人工智能项目时，无论你是否处于正式的领导岗位，带领项目前进的能力都会变得更加重要。这种能力不仅有助于个人成长为领导者，也对项目的成功有着显著的帮助。
网络在人工智能中的重要性是什么？
在网络和人工智能领域，建立一个强大的专业网络对于推动你的职业发展非常重要。虽然直接的网络建设可能对某些人来说不太舒适，但通过参与社区活动、与同行交流以及进行信息性面试等方式，可以间接地扩展你的人脉。这些人际关系不仅能够提供宝贵的信息，还能在你需要帮助或建议时为你提供支持。此外，当你在寻找工作机会时，这些联系人可能会向潜在雇主推荐你，从而增加你获得职位的机会。因此，在人工智能领域中，网络是个人成长和发展不可或缺的一部分。
要想事业成功，有哪些好习惯？
要想事业成功，可以培养以下几个好习惯：

1. 持续学习：通过阅读、上课或与领域专家交流来不断获取新知识和灵感。这有助于你产生新的想法，并且能够跟上行业的发展。

2. 专注于某个应用领域：选择一个特定的应用领域深入研究，尤其是在机器学习等

In [37]:
# 显示当前数据库中所有应用的 Leaderboard 概览（目前只有 Sentence Window）
tru.get_leaderboard(app_ids=[])

,,Answer Relevance,Context Relevance,Groundedness,latency,total_cost
app_name,app_version,,,,,
LlamaIndex_App,base,0.75,0.388889,0.914815,14.326879,0.0


In [38]:
# launches on http://localhost:8501/
# # 启动 TruLens Dashboard，可视化查看 Sentence Window 引擎的评估结果
tru.run_dashboard()

Starting dashboard ...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:51626 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

### 2. Auto-merging retrieval(自动合并检索)

In [7]:
from utils import build_automerging_index
model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
# 使用自定义函数构建或加载 Automerging Index（自动合并索引）。
# 这是另一种优化检索的方法：检索小块，通过上下文分层结构“向上合并”成更大的块供 LLM 使用。
automerging_index = build_automerging_index(
    documents,
    llm,
    embed_model=model_real_path,
    save_dir="merging_index"
)

2026-04-23 22:48:33,788 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a
2026-04-23 22:48:34,173 - INFO - Loading all indices.


In [8]:
from utils import get_automerging_query_engine

# 获取 Automerging 查询引擎。
automerging_query_engine = get_automerging_query_engine(
    automerging_index,
)

In [9]:
# 执行一次查询测试
auto_merging_response = automerging_query_engine.query(
    "如何构建人工智能项目组合?"
)
print(str(auto_merging_response))

2026-04-23 22:49:27,556 - INFO - > Merging 1 nodes into parent node.
> Parent node id: d8f543a4-a478-46e1-8896-cff9d52ec951.
> Parent node text: PAGE 14
Scoping Successful 
AI Projects
CHAPTER 4
PROJECTS

2026-04-23 22:49:27,557 - INFO - > Merging 1 nodes into parent node.
> Parent node id: dd27fc32-0143-43e0-8d1e-eb82536eb0a0.
> Parent node text: PAGE 24
A Simple Framework 
for Starting Your AI 
Job Search
CHAPTER 7
JOBS

2026-04-23 22:49:27,557 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 448f1c76-5403-481e-a6eb-327a8470d2d5.
> Parent node text: PAGE 37
Overcoming Imposter 
Syndrome
CHAPTER 11

2026-04-23 22:49:27,557 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 5983ea5d-c501-49e2-8d10-51b620337757.
> Parent node text: PAGE 31
Finding the Right 
AI Job for You
CHAPTER 9
JOBS

2026-04-23 22:49:27,558 - INFO - > Merging 1 nodes into parent node.
> Parent node id: 3a7af91d-c7f6-4752-ab94-33283f414150.
> Parent node text: PAGE 41

2026-04-23 22:49:27,5

> Merging 1 nodes into parent node.
> Parent node id: d8f543a4-a478-46e1-8896-cff9d52ec951.
> Parent node text: PAGE 14
Scoping Successful 
AI Projects
CHAPTER 4
PROJECTS

> Merging 1 nodes into parent node.
> Parent node id: dd27fc32-0143-43e0-8d1e-eb82536eb0a0.
> Parent node text: PAGE 24
A Simple Framework 
for Starting Your AI 
Job Search
CHAPTER 7
JOBS

> Merging 1 nodes into parent node.
> Parent node id: 448f1c76-5403-481e-a6eb-327a8470d2d5.
> Parent node text: PAGE 37
Overcoming Imposter 
Syndrome
CHAPTER 11

> Merging 1 nodes into parent node.
> Parent node id: 5983ea5d-c501-49e2-8d10-51b620337757.
> Parent node text: PAGE 31
Finding the Right 
AI Job for You
CHAPTER 9
JOBS

> Merging 1 nodes into parent node.
> Parent node id: 3a7af91d-c7f6-4752-ab94-33283f414150.
> Parent node text: PAGE 41

> Merging 1 nodes into parent node.
> Parent node id: 27555b48-eb0a-4cb3-b2d9-c842c9e568b6.
> Parent node text: PAGE 14
Scoping Successful 
AI Projects
CHAPTER 4
PROJECTS

> Merging 1 no

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-23 22:49:35,401 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


构建人工智能项目组合时，首先需要明确项目的范围和目标。这包括识别是否有合适的人工智能解决方案来满足特定需求。如果发现没有合适的AI解决方案，接受这一点也是很重要的。接下来，可以按照一定的步骤来进行，比如先定义问题，再确定解决问题的方法等。确保每个项目都有清晰的目标，并且这些目标是实际可行的。通过这种方式，你可以逐步建立起一个成功的人工智能项目组合。


In [13]:
from utils import get_prebuilt_trulens_recorder
# 准备开始 Automerging 引擎的批量评估
# 注意：这里再次 reset_database() 会清除 Sentence Window 的记录！
# 在实际对比中，通常只在所有测试完成后再调用一次 Leaderboard/Dashboard，而非清除重来。
# 假设这里重置是为了确保 Automerging 的评估从干净的数据库开始，便于观察其自身的性能。
tru.reset_database()

# 为 Automerging 查询引擎创建 TruLens 记录器
tru_recorder_automerging = get_prebuilt_trulens_recorder(automerging_query_engine,
                                                         app_id="Automerging Query Engine")

Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]

instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.embeddings.multi_modal_base.MultiModalEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base

/Users/a1-6/miniconda3/envs/ailearn/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 5 fields but got 4: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='{\n  "cr...one, function_call=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ne, function_call=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/Users/a1-6/miniconda3/envs/ailearn/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 5 fields but got 4: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='{\n  "cr...one, function_call=None), input_type=Mess

In [18]:
# 使用 TruLens 记录器对所有评估问题进行批量查询和评估
for question in eval_questions:
    with tru_recorder_automerging as recording:
        response = automerging_query_engine.query(question)
        print(question)
        print(response)

> Merging 1 nodes into parent node.
> Parent node id: 172eebd3-1f02-4669-822d-244c47a50ae1.
> Parent node text: PAGE 2
"AI is the new 
electricity. It will 
transform and improve 
all areas of human life."
And...

> Merging 1 nodes into parent node.
> Parent node id: 90ddd55e-ab07-4e5b-a920-bce794ac71f3.
> Parent node text: PAGE 2
"AI is the new 
electricity. It will 
transform and improve 
all areas of human life."
And...

在人工智能领域建立职业生涯的关键是什么？
在人工智能领域建立职业生涯的关键包括技术和人际关系两个方面。技术上，选择能够让你成长的项目非常重要，理想的项目应该具有一定的挑战性，可以提升你的技能，但又不至于难到让你几乎没有成功的可能。这样可以帮助你逐步掌握更复杂的技术。

同时，与优秀的团队成员合作也非常关键。良好的合作者对你的成长有着巨大的影响。如果当前没有好的队友，寻找可以讨论问题的人也很重要，因为我们能从周围的人身上学到很多东西。

此外，建立和维护一个强大的人际网络也非常重要。你认识的人不仅可以提供宝贵的信息，还可以为你推荐潜在的雇主，从而帮助你在职业道路上取得更大的成功。
> Merging 1 nodes into parent node.
> Parent node id: 172eebd3-1f02-4669-822d-244c47a50ae1.
> Parent node text: PAGE 2
"AI is the new 
electricity. It will 
transform and improve 
all areas of human life."
And...

> Merging 1 nodes into parent node.
> Parent node id: 90ddd55e-ab

In [20]:
# 显示当前数据库中所有应用的 Leaderboard 概览
tru.get_leaderboard(app_ids=[])

,,Answer Relevance,Context Relevance,Groundedness,latency,total_cost
app_name,app_version,,,,,
LlamaIndex_App,base,1.0,0.166667,0.666667,8.152154,0.0


In [21]:
# 启动 TruLens Dashboard，可视化查看 Automerging 引擎的评估结果
tru.run_dashboard()

Starting dashboard ...


/var/folders/zt/nbrxt1lj4g7gf5l2hqmsq4m00000gp/T/ipykernel_19148/2291147825.py:2: DeprecationWarning: Method `run_dashboard` has been renamed or moved to `trulens.dashboard.run.run_dashboard`.

  tru.run_dashboard()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:62944 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

In [22]:
tru.stop_dashboard()

/var/folders/zt/nbrxt1lj4g7gf5l2hqmsq4m00000gp/T/ipykernel_19148/2300421333.py:1: DeprecationWarning: Method `stop_dashboard` has been renamed or moved to `trulens.dashboard.run.stop_dashboard`.

  tru.stop_dashboard()
